In [1]:
# 라이브러리 할당
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [2]:
# csv 파일 불러오기
df = pd.read_csv('ml.csv')

df

,title,bot,minor,length_old,length_new,len_diff,time_delta_sec,is_first_post,type_edit,type_log,...,title_len,is_revert,is_update,is_minor_fix,comment_len,user_activity_score,is_pro_editor,rev_diff,is_section_edit,comment_clean_len
0,0,False,1,77336,77336,0,0.0,True,True,False,...,1,0,0,1,42,1,0,0,1,39
1,112th Grey Cup,False,1,52533,52566,33,0.0,True,True,False,...,14,0,0,0,99,51,1,0,0,84
2,13th Illinois General Assembly,False,1,24570,24571,1,0.0,True,True,False,...,30,0,0,1,57,28,0,0,1,54
3,13th Parliament of British Columbia,False,1,12690,12690,0,0.0,True,True,False,...,35,0,0,1,60,28,0,0,1,57
4,13th Tripura Assembly,False,0,19183,19427,244,0.0,True,True,False,...,21,0,0,0,37,4,0,0,1,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3221,Zee Asadel,False,0,19958,20203,245,0.0,True,True,False,...,10,0,0,0,10,1,0,0,1,5
3222,Zenande Mfenyana,False,0,4988,5470,482,0.0,True,True,False,...,16,0,1,0,17,2,0,0,0,17
3223,Zhang Dingfan,True,1,5915,5931,16,0.0,True,True,False,...,13,0,0,0,43,37,0,0,0,43
3224,Étienne-Charles de Damas-Crux,True,1,8310,8342,32,0.0,True,True,False,...,29,0,0,0,50,37,0,0,0,50


In [3]:
# 컬럼 확인하기
print(df.columns)

Index(['title', 'bot', 'minor', 'length_old', 'length_new', 'len_diff',
       'time_delta_sec', 'is_first_post', 'type_edit', 'type_log', 'type_new',
       'ns_grouped_1', 'ns_grouped_14', 'ns_grouped_2', 'ns_grouped_3',
       'ns_grouped_4', 'ns_grouped_others', 'title_len', 'is_revert',
       'is_update', 'is_minor_fix', 'comment_len', 'user_activity_score',
       'is_pro_editor', 'rev_diff', 'is_section_edit', 'comment_clean_len'],
      dtype='object')


### [STEP 1] 분석용 핵심 변수 생성

In [5]:
# time__delta_sec 역수 처리 (작을수록 중요하므로, 값이 클수록 중요하게 변환)
# 분모 0 방지를 위해 +1
df['time_inv'] = 1/(df['time_delta_sec'] + 1)

# isolation Forest와 Linear Regressino이 학습을 잘 할 수 있도록 '수치형' 데이터만 선택
# 숫자형인 컬럼만 자동으로 수집
features = df.select_dtypes(include=[np.number]).columns.tolist()

# 분석에 방해되는 식별자나 타켓 후보는 제거 (length_old, length_new, time_delate_sec)
to_exclude = ['length_old', 'length_new', 'time_delta_sec']
features = [f for f in features if f not in to_exclude]

In [6]:
# X 값 할당
X = df[features]

In [7]:
# 데이터 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### [STEP 2] Isolation Forest로 '종합 이상치 점수(KEI)' 산출

In [9]:
# 모든 변수를 고려했을 때 '이 편집은 정말 특이하다!' 라는 점수 뽑음
model_if = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_score'] = model_if.fit(X_scaled).decision_function(X_scaled) * -1

In [10]:
# 점수 정규화 (0~1)
df['target_kei'] = (df['anomaly_score'] - df['anomaly_score'].min()) / \
                   (df['anomaly_score'].max() - df['anomaly_score'].min())

### [STEP 3] Linear Regression으로 모든 변수의 가중치 역산

In [12]:
model_lr = LinearRegression()
model_lr.fit(X, df['target_kei'])

LinearRegression()

In [13]:
# 모든 변수의 가중치를 데이터프레임으로 정리
importance_df = pd.DataFrame({
    'Feature': features,
    'Weight': model_lr.coef_
}).sort_values(by='Weight', ascending=False)

In [14]:
print("📊 모든 컬럼을 활용한 KEI 기여도")
print(importance_df)

📊 모든 컬럼을 활용한 KEI 기여도
                Feature        Weight
3             is_revert  3.595142e-01
5          is_minor_fix  2.786735e-01
0                 minor  2.620459e-01
8         is_pro_editor  2.050355e-01
10      is_section_edit  1.337276e-01
4             is_update  5.004448e-02
11    comment_clean_len  1.944122e-03
7   user_activity_score  5.085153e-04
2             title_len  3.590066e-04
1              len_diff  4.205758e-07
9              rev_diff -2.220446e-16
6           comment_len -6.319398e-04
12             time_inv -7.068067e-02


In [16]:
# =================================================================
# [결과 적용] 도출된 가중치를 활용한 최종 KEI 산출
# =================================================================

# importance_df를 딕셔너리로 변환하여 가중치 맵핑
weights = importance_df.set_index('Feature')['Weight'].to_dict()

# 모든 변수에 가중치를 곱하여 KEI 합산
df['KEI_final'] = 0
for feature, weight in weights.items():
    df['KEI_final'] += df[feature] * weight

# 가장 핫한 문서 TOP 20 확인
print(df.groupby('title')['KEI_final'].mean().sort_values(ascending=False).head(20))

title
Draft:The Monkeys 3                              1.047647
Bernard de Nogaret                               0.751262
User:Benmite/Acacia Brinley                      0.732537
Barnlund's model of communication                0.721122
1889 Newfoundland general election               0.718856
6th National Assembly of Azerbaijan              0.712654
13th Parliament of British Columbia              0.703469
1935–36 Midwest Basketball Conference season     0.702764
Association of the United States Army            0.701563
2003 Pittsburgh Steelers season                  0.700720
2007 West Virginia Mountaineers football team    0.700498
Anandpur Sahib Resolution                        0.698567
17th Illinois General Assembly                   0.697737
13th Illinois General Assembly                   0.697737
Apophatic theology                               0.696054
Atlantis (video game)                            0.695818
Archibald McIntyre (carpenter)                   0.695113
1968 Hou

In [18]:
df.to_csv('ml2.csv', index=False, encoding='utf-8')